# 01 — The end-to-end workflow (a linear classifier)

Start with the **simplest possible model** so the focus is the *workflow*, not the architecture: make data → split train/val → define a model → train loop → **evaluate** → **plot results**. Every later notebook reuses this exact skeleton with a fancier model.

Model: a single `nn.Linear(2, 3)` — a linear (multinomial-logistic) classifier. Data: three 2-D Gaussian blobs, which are linearly separable, so a linear model suffices.

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## Data — three blobs, split into train/val

A held-out validation set is how you measure *generalization* (performance on data not trained on), separate from *fitting* the training data.

In [ ]:
def make_blobs(n, k=3):
    centers = np.array([[0, 0], [3, 3], [-3, 3]], dtype=np.float32)
    y = np.random.randint(0, k, n)
    X = centers[y] + np.random.randn(n, 2).astype(np.float32) * 0.8
    return torch.tensor(X), torch.tensor(y)

Xtr, ytr = make_blobs(1500)
Xva, yva = make_blobs(500)
Xtr, ytr, Xva, yva = (t.to(device) for t in (Xtr, ytr, Xva, yva))
print("train:", Xtr.shape, "val:", Xva.shape)

plt.figure(figsize=(4, 4))
plt.scatter(Xtr[:, 0].cpu(), Xtr[:, 1].cpu(), c=ytr.cpu(), s=8, cmap="tab10")
plt.title("training data (3 blobs)"); plt.show()

## Model, loss, optimizer

`nn.Linear(2, 3)` maps a 2-D point to 3 class logits. `F.cross_entropy` takes raw logits + integer labels. `AdamW` updates the parameters.

In [ ]:
model = nn.Linear(2, 3).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
print("params:", sum(p.numel() for p in model.parameters()))

## The training loop + evaluation

The reusable pattern: for each epoch, run the 5-line train step over mini-batches; then in `eval()` + `no_grad()` measure loss and accuracy on **both** splits. We record them to plot learning curves.

In [ ]:
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}
bs = 64
for epoch in range(40):
    model.train()
    perm = torch.randperm(len(Xtr))
    for k in range(0, len(Xtr), bs):
        idx = perm[k:k+bs]
        loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    tl, ta = evaluate(model, Xtr, ytr); vl, va = evaluate(model, Xva, yva)
    hist["tr_loss"].append(tl); hist["va_loss"].append(vl)
    hist["tr_acc"].append(ta); hist["va_acc"].append(va)

print(f"final  train acc {hist['tr_acc'][-1]:.3f}  |  val acc {hist['va_acc'][-1]:.3f}")

## See the results — learning curves

Loss should fall and accuracy rise on both splits, tracking together (no gap) — the sign of a well-fit, non-overfit model.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="val")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist["tr_acc"], label="train"); ax[1].plot(hist["va_acc"], label="val")
ax[1].set_title("accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## Takeaways

- The **workflow** — data → split → model → train loop → eval → plot — is identical for every model; only the model and data change.
- **Train vs. val**: fitting is train performance; *generalization* is val performance. Watch both.
- **`eval()` + `no_grad()`** for measurement (no dropout/grad bookkeeping).
- Curves are how you *see* training — always plot loss and a metric.

Next: a linear model can only draw straight boundaries. Notebook 02 uses data where that fails, and adds depth + nonlinearity — then shows overfitting and how to control it.